In [ ]:
#以流域为单位计算nc的平均值，注意修改变量名称×1
import os
import pandas as pd
import xarray as xr
import geopandas as gpd

# 输入文件夹和输出路径
shp_folder = r"xxx\BasinATLAS5"  # .shp文件所在的文件夹
nc_folder = r"E:\database\soil-POLES-data\soil\soil-high\\SILT"  # .nc文件所在的文件夹
csv_output_folder = r"xxx\\SILT"  # 输出CSV文件的文件夹路径

# 指定要处理的变量名
target_variable = "SILT"  # !!!!!!!!!!!!此处修改变量名称  

# 读取所有shp文件和nc文件
shp_files = [f for f in os.listdir(shp_folder) if f.endswith('.shp')]
nc_files = [f for f in os.listdir(nc_folder) if f.endswith('.nc')]

# 检查是否有shp文件存在
if len(shp_files) == 0:
    raise ValueError("No shapefile found in the provided directory.")

# 遍历每个shp文件
for shp in shp_files:
    shp_path = os.path.join(shp_folder, shp)  # 获取当前shp文件的完整路径
    shapefile = gpd.read_file(shp_path)  # 使用Geopandas读取矢量文件

    # 初始化结果列表
    results = []

    try:
        # 遍历所有.nc文件
        for nc in nc_files:
            nc_path = os.path.join(nc_folder, nc)  # 获取.nc文件的完整路径
            dataset = xr.open_dataset(nc_path)  # 打开NetCDF文件

            # 检查指定变量是否存在
            if target_variable not in dataset.variables:
                print(f"Variable '{target_variable}' not found in {nc}. Skipping this file.")
                continue

            # 提取目标变量
            variable_data = dataset[target_variable]

            # 获取.nc文件的经纬度范围
            lon = dataset['lon']  # 经度变量
            lat = dataset['lat']  # 纬度变量

            # 将shp的边界框转换为裁剪范围
            bounds = shapefile.total_bounds  # 获取shapefile的边界框
            lon_min, lat_min, lon_max, lat_max = bounds

            # 裁剪 NetCDF 数据
            clipped_data = variable_data.where(
                (lon >= lon_min) & (lon <= lon_max) &
                (lat >= lat_min) & (lat <= lat_max), drop=True
            )

            # 检查是否裁剪成功，避免尺寸为0的情况
            if clipped_data.size == 0:
                print(f"Skipping {nc} for {shp}: No overlap.")
                continue

            # 计算裁剪后的均值
            try:
                mean_value = clipped_data.mean().item()
            except ValueError:
                print(f"Cannot apply mean to {shp} with {nc} due to empty or invalid data. Skipping this file.")
                continue  # 跳过当前 .nc 文件

            # 提取.nc文件名中的数字部分
            nc_number = ''.join(filter(str.isdigit, nc))

            # 存储结果
            results.append([shp, nc_number, mean_value])

            print(f"Processed: {shp} with {nc} - MEAN for {target_variable}: {mean_value}")

    except Exception as e:
        print(f"Skipping {shp} due to an error: {e}")
        continue  # 如果遇到无法处理的错误，跳过当前 .shp 文件

    # 生成输出CSV文件路径
    if results:
        output_csv = os.path.join(csv_output_folder, os.path.splitext(shp)[0] + ".csv")

        # 将结果保存到CSV
        df = pd.DataFrame(results, columns=["Shapefile", "NC Number", "Mean Value"])
        df.to_csv(output_csv, index=False)

        print(f"Results for {shp} saved to: {output_csv}")
    else:
        print(f"No valid data for {shp}. Skipping saving results.")

print("Processing complete!")


Processed: element_0.shp with SILT5min.nc - MEAN for SILT: 25.111148834228516
Results for element_0.shp saved to: E:\\future\\Kmeans\5\\SILT\element_0.csv
Processed: element_1.shp with SILT5min.nc - MEAN for SILT: 24.831911087036133
Results for element_1.shp saved to: E:\\future\\Kmeans\5\\SILT\element_1.csv
Processed: element_10.shp with SILT5min.nc - MEAN for SILT: 34.7237548828125
Results for element_10.shp saved to: E:\\future\\Kmeans\5\\SILT\element_10.csv
Processed: element_100.shp with SILT5min.nc - MEAN for SILT: 22.297710418701172
Results for element_100.shp saved to: E:\\future\\Kmeans\5\\SILT\element_100.csv
Processed: element_1000.shp with SILT5min.nc - MEAN for SILT: 16.284635543823242
Results for element_1000.shp saved to: E:\\future\\Kmeans\5\\SILT\element_1000.csv
Processed: element_1001.shp with SILT5min.nc - MEAN for SILT: 23.80137062072754
Results for element_1001.shp saved to: E:\\future\\Kmeans\5\\SILT\element_1001.csv
Processed: element_1002.shp with SILT5min.nc -